## Training notebook

In [1]:
import gym
import highway_env

from stable_baselines3 import PPO
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact
import ipywidgets as widgets
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_checker import check_env
from highway_env.tb_callback import TensorboardCallback

from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UBUNTU_PIGO','WINDOWS_FORNO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('UBUNTU_PIGO', 'WINDOWS_FORNO', 'UBUNTU_DELL…

<function __main__.f(desired_os)>

In [3]:
if os_id.value == 'UBUNTU_DELL':
    OUTPUT_PATH = '/home/elios/pighetti/HighwayDRL/training_output'
elif os_id.value == 'UBUNTU_PIGO':
    OUTPUT_PATH = '/home/pigo/HighwayDRL/training_output'
elif os_id.value == 'WINDOWS_FORNO':
    OUTPUT_PATH = r'C:/Users/luka-/OneDrive/Documenti/PhD/HighwayDRL/training_output'

### Select environment

In [4]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['multipleO-dm-env-v0','highway-v0','dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('multipleO-dm-env-v0', 'highway-v0', 'dm-env-…

<function __main__.f(input_env)>

In [5]:
total_timesteps = 2e5
# Create and wrap the environment
env = gym.make(env_id.value)
env = Monitor(env, filename='../../training_output/log') 
check_env(env)
env.configure({
    "training_total_timesteps": total_timesteps
})

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback_CONS')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

tb_callback = TensorboardCallback()

/home/elios/miniconda3/envs/APleurent/lib/python3.8/site-packages/stable_baselines3/common/env_checker.py:190: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [6]:
ppo_ilr = 3.5e-4
# PPO parameters
model = PPO("MlpPolicy",
        env,
        seed=123,
        gamma=0.997,
        batch_size=128,
        n_steps=4096,
        n_epochs=10,
        ent_coef=0.1,
        verbose=0,
        learning_rate=ppo_ilr,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

In [7]:
try:
    # Train the agent for n steps
    model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, tb_callback], progress_bar=True)
finally:
    # Save the trained agent
    model.save(f'{OUTPUT_PATH}/models/ppo_3lane_nocurriculum_conservative_200k.zip')

Output()

## Continued learning

In [6]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)
env_cl = Monitor(env_cl, filename='../../training_output/log_cl') 
check_env(env_cl)

/home/elios/miniconda3/envs/APleurent/lib/python3.8/site-packages/stable_baselines3/common/env_checker.py:190: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [7]:
model_cl = PPO.load(f"{OUTPUT_PATH}/models/ppo_2lane_sparseonly", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}/logdir")

In [8]:
# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback_CL')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

tb_callback = TensorboardCallback()

In [9]:
new_timesteps = 1e5
try:
    model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, eval_callback, tb_callback], reset_num_timesteps=False, progress_bar=True)
finally:
    model_cl.save(f'{OUTPUT_PATH}/models/ppo_3lane_conservative_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Output()

Eval num_timesteps=103400, episode_reward=-0.78 +/- 3.57

Episode length: 53.40 +/- 10.46

New best mean reward!

Eval num_timesteps=104400, episode_reward=1.53 +/- 0.71

...

Eval num_timesteps=202400, episode_reward=2.45 +/- 1.27

Episode length: 60.00 +/- 0.00

Eval num_timesteps=203400, episode_reward=2.81 +/- 0.78

Eval num_timesteps=204400, episode_reward=3.03 +/- 0.56

Episode length: 60.00 +/- 0.00